In [1]:
from langchain_core.documents import Document
from langchain_core.tools import tool
from typing import List, Any
from langchain_core.retrievers import BaseRetriever
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

# Re-rank 모델
# uv add sentence-transformers
rerank_model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")
cross_reranker = CrossEncoderReranker(model=rerank_model, top_n=2)

# 웹 검색 리트리버 클래스 정의
class WebSearchRetriever(BaseRetriever):
    search_wrapper: DuckDuckGoSearchAPIWrapper

    def __init__(self, search_wrapper: DuckDuckGoSearchAPIWrapper, **kwargs: Any):
        super().__init__(search_wrapper=search_wrapper, **kwargs)

    def _get_relevant_documents(self, query: str, *, run_manager: CallbackManagerForRetrieverRun) -> List[Document]:
        results = self.search_wrapper.results(query, max_results=10)
        if not results:
            return [Document(page_content="관련 정보를 찾을 수 없습니다.")]
        
        formatted_docs = []
        for result in results:
            doc = Document(
                page_content=f'<Document href="{result.get("link", "")}"/>\n{result.get("snippet", "")}\n</Document>',
                metadata={
                    "source": "web search", 
                    "url": result.get("link", ""), 
                    "title": result.get("title", "")
                }
            )
            formatted_docs.append(doc)
        return formatted_docs

@tool
def web_search(query: str, search_period: str = 'm') -> List[Document]:
    """
    웹 검색을 수행하고 결과를 Document 리스트 형태로 반환하는 함수.
    search_period: 검색 기간 (d: 1일, w: 1주, m: 1달, y: 1년)
    """
    # WebSearchRetriever를 기반으로 ContextualCompressionRetriever를 초기화합니다.
    ddg_search_wrapper = DuckDuckGoSearchAPIWrapper(time=search_period)
    web_retriever = ContextualCompressionRetriever(
        base_compressor=cross_reranker, 
        base_retriever=WebSearchRetriever(search_wrapper=ddg_search_wrapper), 
    )

    docs = web_retriever.invoke(query)

    if len(docs) > 0:
        return docs
    
    return [Document(page_content="관련 정보를 찾을 수 없습니다.")]

In [2]:
# 도구 목록을 정의 
tools = [web_search]

In [5]:
query = "KT 소액 결제 해킹 사건의 원인과 대책?"

tools[0].invoke(query)

[Document(metadata={'source': 'web search', 'url': 'https://reasonablegift.tistory.com/175', 'title': 'Kt 소액결제 해킹 사건 정리 (2025) - 피해 현황·원인·보상 절차 완벽 가이드'}, page_content='<Document href="https://reasonablegift.tistory.com/175"/>\n2025년 8월 수도권에서 발생한 KT 소액결제 해킹 사건은 불법 초소형 기지국을 통한 역대급 해킹 피해 사례입니다. 피해 현황, 해킹 원인, KT 및 정부 대응, 보상 절차와 예방법까지 한눈에 정리했습니다.2025년 8월 말부터 수도권 일부 지역을 중심으로 KT 및 KT 망을 사용하는 알뜰폰 고객들에게 휴대폰 ...\n</Document>'),
 Document(metadata={'source': 'web search', 'url': 'https://data0426.com/entry/KT-해킹-사태-정리｜소액결제-피해·개인정보-유출-원인과-대응-방법', 'title': 'Kt 해킹 사태 정리｜소액결제 피해·개인정보 유출 원인과 대응 방법'}, page_content='<Document href="https://data0426.com/entry/KT-해킹-사태-정리｜소액결제-피해·개인정보-유출-원인과-대응-방법"/>\nKT 소액결제·유심(IMSI) 유출 사건: 원인·피해·대응(완전 정리) 핵심 요약 최근 KT 가입자 일부에서 발생한 무단 소액결제 사건과 관련해, KT 는 자체 조사 결과 약 5,561건의 유심 식별 정보(IMSI) 유출 정황을 확인했다고 발표했습니다. 동시에 소액결제 피해는 수백 건, 피해액 약 1억7천만 원대로 ...\n</Document>')]